In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Read in clean BBHI senior data
bbhi_senior = pd.read_csv("~/Documents/2023:2024/Data/Exported data/clean_bbhi_senior_tp1.csv")
bbhi_senior = bbhi_senior.apply(lambda col: col.astype(float) if col.name not in ["id", "sex"] else col)

print(f"Number of BBHI senior participants: {len(bbhi_senior)}")

# Read BBHI education data
bbhi = pd.read_csv("~/Documents/2023:2024/Data/Exported data/clean_bbhi_tp1.csv")
bbhi = bbhi.apply(lambda col: col.astype(float) if col.name not in ["id", "sex"] else col)

print(f"Number of BBHI participants: {len(bbhi)}")

# Identify common columns to merge on
common_columns = [col for col in bbhi.columns if col in bbhi_senior.columns]

# Merge on all common columns
data = pd.merge(bbhi, bbhi_senior, on=common_columns, how="outer")

# Filter to only include participants with w2_age data
data = data[data["w2_age"].notna()]

# Print the number of participants after filtering
print(f"Number of participants: {len(data)}")

data.sample(5)

In [ ]:
# Look at data for available cross-sectional analysis
participants = data["w1_age"].count()
mean_age = data["w1_age"].mean()
std_dev_age = data["w1_age"].std()
min_age = data["w1_age"].min()
max_age = data["w1_age"].max()

print(f"Number of participants: {participants}")
print(f"Mean age: {mean_age:.2f}")
print(f"SD age: {std_dev_age:.2f}")
print(f"Range age: {min_age:.2f} - {max_age:.2f}")


In [ ]:
# Readin the Neuronorma data (from the two publication above) in excel form & put into a df
xls = pd.ExcelFile("/Users/rachelmorse/superagers/classification/neuronorma_neuropsych_data.xlsx")
score_mappings = {sheet_name: pd.read_excel(xls, sheet_name) for sheet_name in xls.sheet_names}

In [ ]:
# Round down years of education data because some participants have data that is not a whole number (e.g. YoE = 9.5)
# This gives participants the lower education value because they will have their cognitive test scores normalized according to education level, disadvantaging them if it is rounded up
data["YoE"] = np.floor(data["YoE"])

# Round the ages of the participants to the nearest whole number because you need a whole number for calulating the norms
data["w1_age_round"] = np.round(data["w1_age"])
data["w2_age_round"] = np.round(data["w2_age"])

In [ ]:
# Function to create scaled scores for that adjust for age
def map_raw_to_scaled_age(raw_score, age, var):
    """Maps raw scores to scaled TMT-B scores based on age.

    Args:
        raw_score (int): Raw neuropsych score for the participant
        age (int): Age of the participant
        var (str): Variable name for the raw score in the score_mappings DataFrame
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(int, age_range.split("-"))  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row[var])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None

In [ ]:
def map_raw_to_scaled_edu(data, excel_path, sheet_name,
                   norm_age_var, 
                   out_col="norm_score", max_yoe=20):
    """Adds scaled scores from a Neuronorma table adjusted for education.

    Args:
        data (pd.DataFrame): The neuropsych dataset.
        excel_path (str): Path to Neuronorma Excel file.
        sheet_name (str): Sheet name for the specific test (e.g., "TMTB", "SemanticFluency").
        norm_age_var (str): Column in neuropsych dataset with normative age index.
        out_col (str): Name of output column for the scaled scores.
        max_yoe (int): Cap for years of education in the lookup table.
    """
    # Load the normative lookup table
    edu_table = pd.read_excel(excel_path, sheet_name=sheet_name, index_col=0)

    def lookup(row):
        try:
            age_idx = int(row[norm_age_var])
            yoe = int(min(row["YoE"], max_yoe))
            return edu_table.loc[age_idx, yoe]
        except Exception:
            return np.nan

    data[out_col] = data.apply(lookup, axis=1)
    return data

In [ ]:
# Create scaled scores for TMT-B that adjust for age and education at tp1
data["w1_tmtb_norm_age"] = data.apply(lambda row: map_raw_to_scaled_age(row["w1_tmt_b_raw"], row["w1_age_round"], "TMTB"), axis=1)

data = map_raw_to_scaled_edu(
    data,
    excel_path="/Users/rachelmorse/superagers/classification/neuronorma_education_data.xlsx",
    sheet_name="TMTB",
    norm_age_var="w1_tmtb_norm_age",
    out_col="w1_tmtb_norm"
)

# Repeat at tp2
data["w2_tmtb_norm_age"] = data.apply(lambda row: map_raw_to_scaled_age(row["w2_tmt_b_raw"], row["w2_age_round"], "TMTB"), axis=1)

data = map_raw_to_scaled_edu(
    data,
    excel_path="/Users/rachelmorse/superagers/classification/neuronorma_education_data.xlsx",
    sheet_name="TMTB",
    norm_age_var="w2_tmtb_norm_age",
    out_col="w2_tmtb_norm"
)

data[["id", "YoE", "w1_age_round", "w1_tmt_b_raw", "w1_tmtb_norm", "w2_age_round", "w2_tmt_b_raw", "w2_tmtb_norm"]].sample(10)

In [ ]:
# Create scaled scores for inverse digits that adjust for age and education for tp1
data["w1_dsb_norm_age"] = data.apply(lambda row: map_raw_to_scaled_age(row["w1_inverse_digits_raw"], row["w1_age_round"], "DS_B"), axis=1)

data = map_raw_to_scaled_edu(
    data,
    excel_path="/Users/rachelmorse/superagers/classification/neuronorma_education_data.xlsx",
    sheet_name="DS_B",
    norm_age_var="w1_dsb_norm_age",
    out_col="w1_dsb_norm"
)

# Repeat at tp2
data["w2_dsb_norm_age"] = data.apply(lambda row: map_raw_to_scaled_age(row["w2_inverse_digits_raw"], row["w2_age_round"], "DS_B"), axis=1)

data = map_raw_to_scaled_edu(
    data,
    excel_path="/Users/rachelmorse/superagers/classification/neuronorma_education_data.xlsx",
    sheet_name="DS_B",
    norm_age_var="w2_dsb_norm_age",
    out_col="w2_dsb_norm"
)

data[["id", "YoE", "w1_age_round", "w1_inverse_digits_raw", "w1_dsb_norm", "w2_age_round", "w2_inverse_digits_raw", "w2_dsb_norm"]].sample(10)

In [ ]:
# Create scaled scores for semantic fluency that adjust for age and education tp1
data["w1_sf_norm_age"] = data.apply(lambda row: map_raw_to_scaled_age(row["w1_sem_fluency_raw"], row["w1_age_round"], "SF"), axis=1)

data = map_raw_to_scaled_edu(
    data,
    excel_path="/Users/rachelmorse/superagers/classification/neuronorma_education_data.xlsx",
    sheet_name="SF",
    norm_age_var="w1_sf_norm_age",
    out_col="w1_sf_norm"
)

# Repeat at tp2
data["w2_sf_norm_age"] = data.apply(lambda row: map_raw_to_scaled_age(row["w2_sem_fluency_raw"], row["w2_age_round"], "SF"), axis=1)

data = map_raw_to_scaled_edu(
    data,
    excel_path="/Users/rachelmorse/superagers/classification/neuronorma_education_data.xlsx",
    sheet_name="SF",
    norm_age_var="w2_sf_norm_age",
    out_col="w2_sf_norm"
)

data[["id", "YoE", "w1_age_round", "w1_sem_fluency_raw", "w1_sf_norm", "w2_age_round", "w2_sem_fluency_raw", "w2_sf_norm"]].sample(10)

In [ ]:
# Calculate who is superager based off of RAVLT score

# Schmidt 1996 - age 16-29 RAVLT-Delayed recall is scoring 12+ (no data adjusted by sex)
data.loc[data["w1_delayed_recall_raw"] >= 12, "w1_superager_ravlt"] = 1
data.loc[data["w1_delayed_recall_raw"] < 12, "w1_superager_ravlt"] = 0

# Repeat at tp2
data.loc[data["w2_delayed_recall_raw"] >= 12, "w2_superager_ravlt"] = 1
data.loc[data["w2_delayed_recall_raw"] < 12, "w2_superager_ravlt"] = 0

# Look at mean and standard deviation of RAVLT scores
mean_ravlt = data["w1_delayed_recall_raw"].mean()
print(f"Mean RAVLT score at tp1: {mean_ravlt:.2f}")

# Print mean age
mean_age = data["w1_age"].mean()
print(f"Mean age of participants at tp1: {mean_age:.2f}")

print("Number of superagers by RAVLT criteria at tp1:")
print(data[data["w1_superager_ravlt"] == 1]["id"].count())

print("Number of superagers by RAVLT criteria at tp2:")
print(data[data["w2_superager_ravlt"] == 1]["id"].count())

# Manually check that everything is running correctly
data[["id", "w1_delayed_recall_raw", "w1_superager_ravlt", "w2_superager_ravlt"]].head(10)

With the scaled scores calculated, the mean is 10 and the SD is 3 for all variables. For the Neuronorma data, see [Peña-Casanova et al. (2009b)](https://pubmed.ncbi.nlm.nih.gov/19549723/) for more info. 

In [ ]:
# Create a superager variable that = 1 when superagers are above 1SD below the norm for TMT-B, semantic fluency, and inverse digits and meet the RAVLT criteria

# Define the variables and the lower bound
variables = ["w1_tmtb_norm", "w1_sf_norm", "w1_dsb_norm", "w2_tmtb_norm", "w2_sf_norm", "w2_dsb_norm"]
lower_bound = 10 - 1 * 3  # Scaled score of 10 is the mean and SD is 3 for all variables

# Create new column 'superager' and initialize it to 0
data["superager"] = 0

# Update 'superager' to 1 for participants who score above the lower bound for TMT-B, semantic fluency, and inverse digits and have 1 for 'superager_RAVLT'
data.loc[(data[variables] >= lower_bound).all(axis=1) & (data["w1_superager_ravlt"] == 1) & (data["w2_superager_ravlt"] == 1), "superager"] = 1

# Display a random sample of 10 rows
sample_df = data.sample(10)
print(data[data["superager"] == 1]["id"].count())

# Display the relevant columns
relevant_columns = ["id", "w1_age_round", "w1_tmtb_norm", "w1_sf_norm", "w1_dsb_norm", "w2_tmtb_norm", "w2_sf_norm", "w2_dsb_norm", "w1_superager_ravlt", "w2_superager_ravlt", "superager"]
sample_df[relevant_columns]

In [ ]:
# Create a new df
clean_df = data

row_count = len(clean_df)
superager_count = clean_df["superager"].sum()
age_matched_controls = row_count - superager_count

print(" ")
print(f"Number of participants: {row_count}")
print(f"Number of superagers: {superager_count:.0f}")
print(f"Number of age-matched controls: {age_matched_controls:.0f}")

# Percentage of superagers
superager_percentage = (superager_count / row_count) * 100
print(f"Percentage of superagers: {superager_percentage:.2f}%")

# Get basic info about superagers
superager_df = clean_df[clean_df["superager"] == 1]

average_age = superager_df["w1_age"].mean()
standard_deviation = superager_df["w1_age"].std()

print(f"Average superager age: {average_age:.2f} (SD {standard_deviation:.2f})")

# Get basic info about controls
controls_df = clean_df[clean_df["superager"] == 0]
average_age = controls_df["w1_age"].mean()
standard_deviation = controls_df["w1_age"].std()

print(f"Average control age: {average_age:.2f} (SD {standard_deviation:.2f})")

clean_df[["id", "superager", "w1_age", "YoE"]].head(15)

In [ ]:
# Filter the df to include only the needed variables
clean_df = clean_df[
    [
        "id",
        "w1_age",
        "YoE",
        "sex",
        "w1_delayed_recall_raw",
        "w1_tmt_b_raw",
        "w1_sem_fluency_raw",
        "w1_tmt_a_raw",
        "w1_direct_digits_raw",
        "w1_inverse_digits_raw",
        "w2_age",
        "w2_delayed_recall_raw",
        "w2_sem_fluency_raw",
        "w2_tmt_b_raw",
        "w2_tmt_a_raw",
        "w2_direct_digits_raw",
        "w2_inverse_digits_raw",
        "w1_crn30",
        "w2_crn30",
        "w1_cro30",
        "w2_cro30",
        "w1_ravlt_total",
        "w2_ravlt_total",
        "w1_crn",
        "w2_crn",
        "w1_cro",
        "w2_cro",
        "superager",
    ]
]

# Export this df to a csv to use for future analysis
clean_df.to_csv("/Users/rachelmorse/Documents/2023:2024/Data/Exported data/superager.csv", index=False)